# 实验 05 · 数据划分与模型选择

目标：建立训练、验证、测试的分工，防止测试集参与选择。

本实验为新增练习，仅依赖 NumPy；题干、测试均不包含参考实现。
从上到下执行，完成每个 TODO 后运行对应自测。

In [1]:
import numpy as np
rng = np.random.default_rng(42)

## 1 · 划分数据索引

实现 split_indices(n,seed)，返回 train_idx、val_idx、test_idx。先随机打乱索引，再按 floor(0.6n)、floor(0.2n)、剩余数量切分；互不重叠，覆盖所有样本，同一 seed 可复现。

In [2]:
def split_indices(n, seed):

    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)

    n_train = int(np.floor(0.6*n))
    n_val = int(np.floor(0.2*n))

    train_idx = idx[ : n_train]
    val_idx = idx[n_train:n_train+n_val]
    test_idx = idx[n_train+n_val:]
    return train_idx, val_idx, test_idx

    raise NotImplementedError("TODO: 无重复的数据划分")

In [3]:
a, b, c = split_indices(23, 42)
assert [len(a), len(b), len(c)] == [13, 4, 6]
np.testing.assert_array_equal(np.sort(np.concatenate([a, b, c])), np.arange(23))
for x, y in ((a,b), (a,c), (b,c)):
    assert not np.intersect1d(x, y).size
for x, y in zip((a,b,c), split_indices(23, 42)):
    np.testing.assert_array_equal(x,y)
assert not np.array_equal(a, np.arange(13))
assert not np.array_equal(a, split_indices(23, 7)[0])
print("通过：比例、完整覆盖、无重叠、可复现")

通过：比例、完整覆盖、无重叠、可复现


## 2 · 只用验证误差选择模型

实现 choose_model(validation_errors)，返回最小误差的索引；并列时选择最早的索引。本函数不能读取测试集。空列表抛出 ValueError。

In [5]:
def choose_model(validation_errors):

    return int(np.argmin(validation_errors))

    raise NotImplementedError("TODO: 验证集模型选择")

In [6]:
assert choose_model([0.30, 0.15, 0.20]) == 1
assert choose_model([0.10, 0.10, 0.20]) == 0
assert choose_model([0.25]) == 0
try:
    choose_model([])
except ValueError:
    pass
else:
    raise AssertionError("空列表必须抛出 ValueError")
print("通过：最小验证误差、并列、空列表")

通过：最小验证误差、并列、空列表


## 3 · 手动诊断偏差与方差

下面三组分别给出训练/验证误差：A=(0.35,0.37)，B=(0.01,0.30)，C=(0.02,0.03)。假设任务可达到约 0.02 的误差。给每组提出一个优先实验并解释理由；说明为什么不能反复看测试误差选择超参数。

In [7]:
# TODO: 以字符串记录你的判断和下一步实验。
diagnosis = """
A=(0.35,0.37)：训练误差和验证误差都远高于目标约 0.02，且两者差距较小，说明主要是高偏差/欠拟合。优先实验是增加模型容量，例如使用更复杂的模型、增加特征或层数，并检查优化过程是否充分。
B=(0.01,0.30)：训练误差很低但验证误差明显偏高，说明主要是高方差/过拟合。优先实验是加入正则化、增加数据或数据增强、早停，或适当简化模型。
C=(0.02,0.03)：训练误差接近目标，验证误差略高，偏差和方差都较小。优先保持当前设置，并做稳健性检查，必要时只根据验证集做小幅超参数调整。
不能反复查看测试误差来选择超参数，否则测试集信息会泄露到模型选择过程中，使测试误差偏乐观，并失去对最终泛化能力的独立估计。测试集应只在最终评估时使用。
"""
# raise NotImplementedError("TODO: 误差诊断")

In [8]:
assert isinstance(diagnosis, str) and len(diagnosis.strip()) > 30
print("格式通过；请自行核对解释是否有证据支持。")

格式通过；请自行核对解释是否有证据支持。


## 实验记录

写出输入/输出形状，解释一个边界情况，再记录一次修改参数前后的观察。

`split_indices(23, 42)` 输入标量 `n=23, seed=42`，输出三个一维整型索引数组，长度分别为 `(13, 4, 6)`，三者拼接排序后恰为 `arange(23)`。`choose_model([0.30, 0.15, 0.20])` 输入长度 3 的一维误差列表，输出标量索引 `1`；空列表抛 `ValueError`。

`n=5`：`floor(0.6n)=3`、`floor(0.2n)=1`，测试集取剩余 `1` 个，仍完整覆盖且无重叠。极端地 `n=1` 时训练/验证为空、测试为 `[0]`，函数不报错，但下游训练需自行校验非空。`choose_model` 对并列最小误差 `[0.10, 0.10, 0.20]` 返回 `0`（最早索引），符合“并列取最早”的约定。

将 `split_indices(23, 42)` 的 seed 改为 `7`：各划分长度仍为 `(13, 4, 6)`、仍完整覆盖无重叠，但具体索引数组不同，说明 seed 只影响划分内容、不影响比例与可复现性。模型选择部分若把验证误差从 `[0.30, 0.15, 0.20]` 改为 `[0.30, 0.15, 0.15]`，返回值由 `1` 变为 `1`（并列取最早），说明并列规则稳定；测试集始终未参与该选择过程。